importation des libs et chargement des datas

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px  # Interface haut niveau pour graphiques simples
import plotly.graph_objects as go  # Interface bas niveau pour contrôle précis
import seaborn as sns
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots  # Création de grilles de graphiques

#Affichage complet des colonnes et des lignes (pas de retour a la ligne)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.expand_frame_repr", False)

#Import du fichier csv customer
url = "./../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url,sep=",",skipinitialspace=True)

# Affichage des dimension du jeu de données
print(f"Le jeu de données a {df.shape[0]} lignes et {df.shape[1]} colonnes")

# Affichez les 5 premières lignes
print(df.head())




Le jeu de données a 7043 lignes et 21 colonnes
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges  TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85         29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.50    No
2  3668-QPYBK    Male              0      No         No       2          Yes 

Affichez les types de données de chaque colonne

Nombre pour chaque variable

In [14]:
# Affichez les types de données de chaque colonne
display(df.info())
print(" " * 50)
print("-" * 50)
print(" " * 50)
#Nombre de manquants pour chaque colonnes

if (df.isna().sum() == 0).all():
    print("Il ne manque aucune valeur.")
else:
    print("Il y a des valeurs manquantes.")
    manquant=df.isna().sum()
    print(f"{manquant[manquant != 0]}")  

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

None

                                                  
--------------------------------------------------
                                                  
Il y a des valeurs manquantes.
TotalCharges    11
dtype: int64


Recherche de doublon

In [6]:
#Recherhce de doublon
print(f"Il y a {df.duplicated().sum()} doublons dans le jeu de données.")
if df.duplicated().sum() > 0:
    print("Voici les lignes en double :")
    print(df[df.duplicated()])

Il y a 0 doublons dans le jeu de données.


Recherche des outlier

In [26]:
def detecter_outliers_iqr(df: pd.DataFrame, colonne: str, coefficient: float = 1.5) -> tuple:
    """
    Détecte les outliers dans une colonne numérique selon la méthode IQR.

    Cette fonction calcule les quartiles Q1 et Q3, puis l'écart interquartile (IQR).
    Elle identifie ensuite les valeurs qui dépassent les limites définies par
    Q1 - coefficient * IQR et Q3 + coefficient * IQR.

    Parameters
    ----------
    df : pd.DataFrame
        Le DataFrame contenant les données à analyser.
    colonne : str
        Le nom de la colonne numérique à analyser.
    coefficient : float, default=1.5
        Le coefficient multiplicateur de l'IQR pour définir les limites.
        La valeur standard est 1.5. Une valeur plus élevée (ex: 3) détectera
        uniquement les outliers extrêmes.

    Returns
    -------
    tuple
        Un tuple contenant :
        - list : Liste triée des index des outliers détectés
        - float : Limite inférieure (valeurs en dessous sont des outliers)
        - float : Limite supérieure (valeurs au-dessus sont des outliers)

    Examples
    --------
    >>> outliers_idx, limite_inf, limite_sup = detecter_outliers_iqr(df, 'Vidéos')
    >>> print(f"Outliers détectés aux index : {outliers_idx}")

    Notes
    -----
    Cette fonction utilise la méthode quantile() de Pandas pour calculer
    les quartiles (gère automatiquement les valeurs NaN).
    """

    # Calcul du 1er quartile (Q1) - 25% des données sont inférieures
    q1 = df[colonne].quantile(0.25)

    # Calcul du 3ème quartile (Q3) - 75% des données sont inférieures
    q3 = df[colonne].quantile(0.75)

    # Calcul de l'écart interquartile (IQR)
    iqr = q3 - q1

    # Calcul des limites
    limite_inferieure = q1 - coefficient * iqr
    limite_superieure = q3 + coefficient * iqr

    # Création d'un masque booléen pour identifier les outliers
    masque_outliers = (df[colonne] < limite_inferieure) | (df[colonne] > limite_superieure)

    # Série des outliers (index + valeur)
    s_outliers = df.loc[masque_outliers, colonne].sort_index()

    # Array 2 colonnes: [index, valeur]
    outliers_array = np.column_stack((s_outliers.index.to_numpy(), s_outliers.to_numpy()))

    return outliers_array, limite_inferieure, limite_superieure

#Recherche des outlier pour chaque colonne numerique

for col in df.select_dtypes(include="number").columns:
    outliers_array, limite_inferieure, limite_superieure= detecter_outliers_iqr(df, col)
    if len(outliers_array) > 0:
    
        # Affichage des résultats
        print(col)
        print(f"Les potentiels outliers (valeurs < {limite_inferieure:.2f} ou > {limite_superieure:.2f}) sont aux index :")
        if outliers_array.size <= 10:
            for idx, val in outliers_array:
                print(f"valeur: {val} | index: {idx}")
        else:
            print (f"il y a {outliers_array.size} outliers sur la colonnne {col}")
            valeurs_outlier = np.unique(outliers_array[:, 1])
            print(f"les valeurs sont:")
            for val in valeurs_outlier:
                print([val])

SeniorCitizen
Les potentiels outliers (valeurs < 0.00 ou > 0.00) sont aux index :
il y a 2284 outliers sur la colonnne SeniorCitizen
les valeurs sont:
[np.int64(1)]


Recherche de incoherances

In [8]:
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())

In [9]:
#Recherche valeur autre que Yes No pour "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"

cols_yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn","MultipleLines"]
valeurs_autorisees = ["Yes", "No","No phone service"]

for col in cols_yes_no:
    s = df[col]
    #selection des valeurs non NaN qui ne sont pas dans les valeurs autorisées
    if col == "MultipleLines":
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees)]
    else:
        #pour les colonnes autre que MultipleLines, les valeurs autorisées sont uniquement "Yes" et "No"
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees[:2])]
        

    
    if invalides.empty:
        print(f"{col}: OK (NaN={s.isna().sum()})")
    else:
        print(f"{col}: INVALIDES -> {invalides.unique()} (NaN={s.isna().sum()})")



Partner: OK (NaN=0)
Dependents: OK (NaN=0)
PhoneService: OK (NaN=0)
PaperlessBilling: OK (NaN=0)
Churn: OK (NaN=0)
MultipleLines: OK (NaN=0)


Recap du controle qualité

In [27]:
# 1) Valeurs manquantes
missing_counts = df.isna().sum()
missing_total = int(missing_counts.sum())

# 2) Doublons (lignes complètes)
duplicates_count = int(df.duplicated().sum())

# 3) outliers
for col in df.select_dtypes(include="number").columns:
    outliers_array, limite_inferieure, limite_superieure= detecter_outliers_iqr(df, col)
    if len(outliers_array) > 0:
    
        # Affichage des résultats
        
        print(f"Les potentiels outliers de la colonne {col} (valeurs < {limite_inferieure:.2f} ou > {limite_superieure:.2f}) sont aux index :")
        if outliers_array.size <= 10:
            for idx, val in outliers_array:
                print(f"valeur: {val} | index: {idx}")
        else:
            print (f"il y a {outliers_array.size} outliers sur la colonnne {col}")
            valeurs_outlier = np.unique(outliers_array[:, 1])
            print(f"les valeurs sont:")
            for val in valeurs_outlier:
                print([val])

# 4) Cohérences métier
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())


# 5) Résumé
print("=== Résumé qualité des données ===")
print(f"Valeurs manquantes totales : {missing_total}")
print(f"Doublons (lignes complètes) : {duplicates_count}")
print(f"Incohérences tenure < 0 : {inco_tenure}")
print(f"Incohérences MonthlyCharges < 0 : {inco_monthly}")


print("\nDétail valeurs manquantes (colonnes avec au moins 1 NaN) :")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

if (
    missing_total == 0
    and duplicates_count == 0
    and inco_tenure == 0
    and inco_monthly == 0
       
    ):
    print("\nConclusion : controles qualite OK.")
else:
    print("\nConclusion : il reste des points a traiter.")

Les potentiels outliers de la colonne SeniorCitizen (valeurs < 0.00 ou > 0.00) sont aux index :
il y a 2284 outliers sur la colonnne SeniorCitizen
les valeurs sont:
[np.int64(1)]
=== Résumé qualité des données ===
Valeurs manquantes totales : 11
Doublons (lignes complètes) : 0
Incohérences tenure < 0 : 0
Incohérences MonthlyCharges < 0 : 0

Détail valeurs manquantes (colonnes avec au moins 1 NaN) :
TotalCharges    11
dtype: int64

Conclusion : il reste des points a traiter.


Exploration de la cible:
    taux Global de churn
    puis par segments 
        Contrat
        InternetService
        tenure
        Payment methode
        gender
        Partner
        Dependents
        PhoneService
        MultipleLines
        InternetService
        OnlineSecurity
        OnlineBackup
        DeviceProtection
        TechSupport
        StreamingTV
        StreamingMovies
        PaperlessBilling
        PaymentMethod
        MonthlyCharges

In [ ]:
#Taux global de churn
taux_global_churn = (df["Churn"] == "Yes").mean()
print(f"Taux global de churn (Pourcentage de personne qui ont résilié) : {taux_global_churn:.2%}")

Definition du protocole d'evalutaion

Split : train / validation / test (stratifié  Churn)
Métriques : ROC-AUC, Precision, Recall, F1-score
Déséquilibre : prise en compte via l’analyse de ces métriques (notamment Recall/F1 sur churn)

In [28]:
def plot_churn_rate_bar(data, segment_col, title=None):
    """
    Calcule et affiche le taux de churn par segment sous forme de bar chart Plotly.

    La fonction s'appuie sur la colonne `Churn_num` (0 = non churn, 1 = churn),
    agrège par `segment_col`, convertit la moyenne en pourcentage, imprime le
    tableau résultant, puis affiche un graphique en barres trié du taux le plus
    élevé au plus faible.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame contenant au minimum les colonnes `Churn_num` et `segment_col`.
    segment_col : str
        Colonne de segmentation utilisée pour le groupement
        (ex. `Contract`, `PaymentMethod`, `tenure_segment`).
    title : str, optional
        Titre personnalisé du graphique. Si `None`, un titre par défaut est utilisé.

    Returns
    -------
    None
        La fonction affiche le tableau agrégé dans la console et le graphique Plotly.
    """
    
    
    
    
    tmp = (
        data.groupby(segment_col, dropna=False)["Churn_num"]
            .mean()
            .mul(100)
            .reset_index(name="churn_rate")
            .sort_values("churn_rate", ascending=False)
    )

    print(f"\n--- {segment_col} ---")
    print(tmp)

    fig = px.bar(
        tmp,
        x=segment_col,
        y="churn_rate",
        color=segment_col,  # une couleur différente par barre
        text=tmp["churn_rate"].round(1).astype(str) + "%",
        labels={segment_col: segment_col, "churn_rate": "Taux de churn (%)"},
        title=title or f"Taux de churn par {segment_col}"
    )

    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        xaxis_tickangle=-25,
        yaxis_range=[0, max(5, tmp["churn_rate"].max() * 1.15)]
    )
    fig.show()


# Churn yes/no -> 0/1
df["Churn_num"] = (df["Churn"]
                    .astype(str).str.strip().str.lower()
                    .map({"yes": 1, "no": 0,"1": 1, "0": 0,"true": 1, "false": 0
                    })
                    )



# Segment tenure
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"]
df["tenure_segment"] = pd.cut(df["tenure"], bins=bins, labels=labels, include_lowest=True)

# Génération de tous les graphes
segments = [
    "tenure_segment",
    "Contract",
    "PaymentMethod",
    "PhoneService",
    "OnlineSecurity",
    "MultipleLines"
]

for col in segments:
    if col in df.columns:
        plot_churn_rate_bar(df, col, f"Taux de churn par {col}")
    else:
        print(f"Colonne absente: {col}")



--- tenure_segment ---
  tenure_segment  churn_rate
0           0-12   47.438243
1          13-24   28.710938
2          25-36   21.634615
3          37-48   19.028871
4          49-60   14.423077
5          61-72    6.609808



--- Contract ---
         Contract  churn_rate
0  Month-to-month   42.709677
1        One year   11.269518
2        Two year    2.831858



--- PaymentMethod ---
               PaymentMethod  churn_rate
2           Electronic check   45.285412
3               Mailed check   19.106700
0  Bank transfer (automatic)   16.709845
1    Credit card (automatic)   15.243101



--- PhoneService ---
  PhoneService  churn_rate
1          Yes   26.709637
0           No   24.926686



--- OnlineSecurity ---
        OnlineSecurity  churn_rate
0                   No   41.766724
2                  Yes   14.611194
1  No internet service    7.404980



--- MultipleLines ---
      MultipleLines  churn_rate
2               Yes   28.609896
0                No   25.044248
1  No phone service   24.926686


Definition du protocole d'evalutaion
   
    Split : train / validation / test (stratifié  Churn), 80,20
    Métriques :  Recall ou precision .
    Déséquilibre : Je vois un déséquilibre avec contrat, methode de payement online security et tenure 